# 第五章 突围战术的语义降维与拓扑结构


## 1. 理论引言：从经验文本到战术拓扑
在第四章验证了“战术平衡区”的病理学机制后，本章聚焦于那些成功存活的“真逃逸（True Flight）”样本。面对庞杂的画廊策展文本与艺术家自述，自然语言本身充满了噪音与修辞的褶皱。
本节运用 TF-IDF 特征提取与 K-Means 无监督聚类，将高维的文本语义降维至二维几何空间。我们的目标不是为了分类，而是为了绘图（Cartography）——在语义的星系中，定位那些距离权力中心最远的“边界样本”，它们往往指示着最具异质性的新生逃逸线。

## 2. 环境配置与语料清洗
载入“真逃逸”样本的文本数据，并进行基础的自然语言预处理。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score

# 绘图审美配置 (Vibe Coding 标准)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

# 载入前置清洗后的数据
df = pd.read_csv("micropolitical_diagnosis_table.csv")
# 仅筛选出判定为“真逃逸”的战术样本
df_true = df[df['Verdict_Type'].astype(str).str.contains('True_Flight', case=False, na=False)].copy()
df_true['cleaned_text'] = df_true['Breakthrough_Strategy'].astype(str).str.replace('_', ' ').str.lower()

print(f"✅ 成功提取有效真逃逸样本：{len(df_true)} 例")

## 3. 特征向量化与最佳 $k$ 值诊断
运用 TF-IDF 将文本转化为稀疏矩阵。为了避免人为预设分类带来的主观暴力，我们交由数据自身涌现其结构，通过轮廓系数（Silhouette Score）和 CH 指数寻找最佳的簇数量 $k$。

In [ ]:
# TF-IDF 特征提取 (提取核心语义元)
vectorizer = TfidfVectorizer(stop_words='english', max_features=100, ngram_range=(1, 2))
X = vectorizer.fit_transform(df_true['cleaned_text'])

# 聚类数诊断 (2 到 8 簇)
K_range = range(2, 9)
sil_scores = []
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=15)
    labels = kmeans.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))

best_k = K_range[np.argmax(sil_scores)]

plt.figure(figsize=(8, 4))
plt.plot(K_range, sil_scores, 'go-', linewidth=2)
plt.axvline(x=best_k, color='red', linestyle='--', alpha=0.7)
plt.title(f"Silhouette Score 分析 (最优 $k$={best_k})")
plt.xlabel("聚类数量 (k)")
plt.ylabel("轮廓系数")
plt.show()

print(f"🤖 算法自动建议的最优战术簇数量：{best_k}")

## 4. 语义空间的 PCA 降维与置信椭圆
在确定 $k$ 值后，实施 K-Means 聚类，并通过主成分分析（PCA）将高维向量投射到 2D 平面，绘制置信椭圆。

In [ ]:
# 执行核心聚类
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=15)
df_true['cluster_id'] = kmeans.fit_predict(X)

# PCA 降维用于可视化
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X.toarray())
df_true['pca_x'], df_true['pca_y'] = X_pca[:, 0], X_pca[:, 1]

# 绘制拓扑散点图 (基于脚本 07 的高级可视化逻辑)
plt.figure(figsize=(10, 8))
colors = plt.cm.tab10.colors

for i in range(best_k):
    subset = df_true[df_true['cluster_id'] == i]
    plt.scatter(subset['pca_x'], subset['pca_y'], s=80, alpha=0.7, label=f'Cluster {i}')
    
# 标注质心
centers_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], marker='X', s=250, c='black', label='Centroid')

plt.title("图 4-4：突围战术的纯数据驱动聚类与语义拓扑", fontsize=14, fontweight='bold')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} 解释方差)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} 解释方差)")
plt.legend()
plt.show()

**哲学合成：寻找“离心”的异质性**
在欧几里得距离下，距离聚类质心（Centroid）最近的样本是该战术流派的“标准范式”；而距离质心最远、处于系统边缘的边界样本（Boundary Samples），则暗示着未被完全结构化的能量。这正是算法对“逃逸线”的数学捕捉。

In [ ]:
# 提取每个簇的距离矩阵
from sklearn.metrics import pairwise_distances
dist_matrix = pairwise_distances(X, kmeans.cluster_centers_)
df_true['dist_to_center'] = dist_matrix[np.arange(len(df_true)), df_true['cluster_id']]

# 提取游离在边界的潜在“逃逸线”样本
print("🚨 边界样本提取 (距离质心最远)：")
for i in range(best_k):
    subset = df_true[df_true['cluster_id'] == i]
    farthest = subset.nlargest(2, 'dist_to_center')
    print(f"\n--- Cluster {i} 的边缘游离者 ---")
    for _, row in farthest.iterrows():
        print(f"ID: {row.get('artwork_id', 'Unknown')} | 距离: {row['dist_to_center']:.3f} | 策略: {row['cleaned_text'][:60]}...")